In [14]:
## ARXIV---research paper ## install arxiv and wikipedia both the libraries
##tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper


In [2]:
##wikipediawrapper
##use the inbuild tools of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)     ##This wrapper will use the Wikipedia API to conduct searches and fetch page summaries. By default, it will return the page summaries of the top-k results. It limits the Document content by doc_content_chars_max.
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)  ##tool creation  ---this is inbuild tool we have
wiki.name


'wikipedia'

In [3]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)  ##This wrapper will use the Arxiv API to conduct searches and fetch research paper summaries. By default, it will return the summaries of the top-k results. It limits the Document content by doc_content_chars_max.
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)  ##tool creation
arxiv.name

'arxiv'

In [4]:
##combine both the tools in a list
tools=[wiki,arxiv]

In [2]:
##lets create custom tools
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain.tools.retriever import create_retriever_tool




USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
loader=WebBaseLoader("https://en.wikipedia.org/wiki/Main_Page")  ##load the data from the web
docs=loader.load()   ##load the data
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)  ##split the data into chunks
vectordb=FAISS.from_documents(documents,embedding=OllamaEmbeddings(model="gemma:2b"))  ##create the vector store
retriever=vectordb.as_retriever()  ##create the retriever
retriever

In [4]:
##convert the retriever into a tool
from langchain.agents import Tool
retriever_tool=create_retriever_tool(retriever=retriever, name="langsmith-search", description="useful for when you")
retriever_tool.name


'langsmith-search'

In [ ]:
tools=[wiki,arxiv,retriever_tool]  ##final tools list

In [ ]:
##run all these tools with agent and llm models
import os
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()

##load the groq api key
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm=ChatGroq(GROQ_API_KE=GROQ_API_KEY,model="llama-3.1-8b-instant")


NameError: name 'api_key' is not defined

In [ ]:
##prompt template
from langchain import hub
#prompt=hub.pull("sunidhi-1234/custom-prompt-template",hub_key="custom-prompt-template")  ##openai function from python.langchain.com website
prompt=hub.pull("hwchase17/openai-function-agent") 
#prompt.messages[0].content
prompt.messages
##🧠 What is hub.pull()?

#LangChain has a Prompt Hub (like GitHub, but for prompts).
#It lets you easily load pre-built, community-contributed prompt templates instead of writing them manually.

In [ ]:
##To exectute all these things in the form of chain we have to use agents   
###there are several years  ##checl tool calling in python.langchain.com
from langchain.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm=llm,tools=tools,prompt=prompt)  ##create the agent 
agent


In [ ]:
## Agent Executer
from langchain.agents import AgentExecutor
agent_executor=AgentExecutor.from_agent_and_tools(agent=agent,tools=tools,verbose=True)
agent_executor 

In [ ]:
agent_executor.run("What is Langchain?")  ##it will call the wikipedia tool and give the answer
## or agent_executor.invoke("{input}: tell me something about langsmith")  ##it will call the arxiv tool and give the answer

In [ ]:
agent_executor.run("What is machine learning?")
